# Hierarchical FSMs

Hierarchical (nested) state machines let a state contain sub-states. A transition on the **parent** applies to every child, which removes duplicated transitions. When the parent is entered, its initial child is entered too.

In the media player, `power_off` is declared once on `on`, yet works from `stopped`, `playing` and `paused`.

```mermaid
stateDiagram-v2
    [*] --> off
    off --> on: power_on
    on --> off: power_off
    state on {
        [*] --> stopped
        stopped --> playing: play
        playing --> paused: pause
        paused --> playing: play
        playing --> stopped: stop
        paused --> stopped: stop
    }
```

In [1]:
from petrilab.fsm.hierarchical import MediaPlayer


def ids(sm):
    return sorted(s.id for s in sm.configuration)


sm = MediaPlayer()
before = ids(sm)
sm.send("power_on")
before, ids(sm)

(['off'], ['on', 'stopped'])

The configuration contains both the parent `on` and its initial child `stopped`: in a hierarchy, several states are active at once (one per level).

## One parent transition, any child

In [2]:
results = {}
for path in [[], ["play"], ["play", "pause"]]:
    sm = MediaPlayer()
    sm.send("power_on")
    for e in path:
        sm.send(e)
    inside = ids(sm)
    sm.send("power_off")
    results[tuple(path)] = (inside, ids(sm))
results

{(): (['on', 'stopped'], ['off']),
 ('play',): (['on', 'playing'], ['off']),
 ('play', 'pause'): (['on', 'paused'], ['off'])}

`power_off` takes the player to `off` from every child state.

## Re-entry resets to the initial child

In [3]:
sm = MediaPlayer()
for e in ("power_on", "play", "power_off", "power_on"):
    sm.send(e)
ids(sm)

['on', 'stopped']

Without history, leaving and re-entering `on` always starts at `stopped`, even though we were `playing` before. (See the history notebook for the alternative.)

## Pause, resume, stop and invalid events

In [4]:
from statemachine.exceptions import TransitionNotAllowed

sm = MediaPlayer()
for e in ("power_on", "play", "pause", "play", "stop"):
    sm.send(e)
print(ids(sm))
try:
    MediaPlayer().send("play")
except TransitionNotAllowed:
    print("play while off is rejected")

['on', 'stopped']
play while off is rejected


The sequence ends in `stopped`. The child events (`play`, `pause`, `stop`) only exist while `on` is active, so `play` is invalid when the player is off.